## **PubMed Sqlite Database Exploration**

In [15]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from bibliometric_rag_assistant.config import DB_PATH, RAW_PUBMED_DIR
from bibliometric_rag_assistant.ingestion.pubmed import (
    fetch_pubmed_citations,
    parse_pubmed_xml,
    search_pubmed,
)
from bibliometric_rag_assistant.search_strategy import (
    PUBMED_ACTIVE_VARIANT,
    PUBMED_QUERY,
    PUBMED_SAMPLE_SIZE,
    PUBMED_SOURCE,
    PUBMED_VARIANTS,
)

In [16]:
def connect_database(db_path: str | Path = DB_PATH) -> sqlite3.Connection:
    # Abre conexão com o arquivo bibliometria.db
    # Banco embutido no disco, sem servidor
    connection = sqlite3.connect(db_path)
    connection.execute("PRAGMA foreign_keys = ON" )

    return connection

In [17]:
def create_tables(connection: sqlite3.Connection) -> None:
    connection.executescript(
        """
        CREATE TABLE IF NOT EXISTS ingestion_runs (
            run_id TEXT PRIMARY KEY,
            source TEXT NOT NULL,
            query_text TEXT NOT NULL,
            query_variant TEXT NOT NULL,
            query_version TEXT NOT NULL,
            retrieved_at TEXT NOT NULL,
            total_results INTEGER NOT NULL,
            total_retrieved INTEGER NOT NULL
        );

        CREATE TABLE IF NOT EXISTS raw_snapshots (
            snapshot_id INTEGER PRIMARY KEY,
            run_id TEXT NOT NULL,
            endpoint TEXT NOT NULL,
            payload_format TEXT NOT NULL,
            payload TEXT NOT NULL,
            file_path TEXT NOT NULL,

            UNIQUE (run_id, endpoint),

            FOREIGN KEY (run_id)
                REFERENCES ingestion_runs (run_id)
        );

        CREATE TABLE IF NOT EXISTS citations (
            run_id TEXT NOT NULL,
            pmid TEXT NOT NULL,
            doi TEXT,
            title TEXT,
            abstract TEXT,
            journal TEXT,
            publication_year INTEGER,
            language TEXT,
            authors_json TEXT NOT NULL,
            affiliations_json TEXT NOT NULL,

            PRIMARY KEY (run_id, pmid),

            FOREIGN KEY (run_id)
                REFERENCES ingestion_runs (run_id)
        );
        """
    )

    connection.commit()

In [18]:
def insert_ingestion_run(
        connection: sqlite3.Connection, 
        run: dict
) -> None:
    connection.execute(
        """
        INSERT INTO ingestion_runs (
            run_id,
            source,
            query_text,
            query_variant,
            query_version,
            retrieved_at,
            total_results,
            total_retrieved
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            run["run_id"],
            run["source"],
            run["query_text"],
            run["query_variant"],
            run["query_version"],
            run["retrieved_at"],
            run["total_results"],
            run["total_retrieved"],
        ),
    )


def insert_raw_snapshot(
    connection: sqlite3.Connection,
    snapshot: dict,
) -> None:
    payload = snapshot["payload"]

    if not isinstance(payload, str):
        payload = json.dumps(
            payload,
            ensure_ascii=False,
        )

    connection.execute(
        """
        INSERT INTO raw_snapshots (
            run_id,
            endpoint,
            payload_format,
            payload,
            file_path
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            snapshot["run_id"],
            snapshot["endpoint"],
            snapshot["payload_format"],
            payload,
            str(snapshot["file_path"]),
        ),
    )


def insert_citations(
    connection: sqlite3.Connection,
    run_id: str,
    citations: list[dict],
) -> None:
    rows = [
        (
            run_id,
            citation["pmid"],
            citation.get("doi"),
            citation.get("title"),
            citation.get("abstract"),
            citation.get("journal"),
            citation.get("publication_year"),
            citation.get("language"),
            json.dumps(
                citation.get("authors", []),
                ensure_ascii=False,
            ),
            json.dumps(
                citation.get("affiliations", []),
                ensure_ascii=False,
            ),
        )
        for citation in citations
    ]

    connection.executemany(
        """
        INSERT INTO citations (
            run_id,
            pmid,
            doi,
            title,
            abstract,
            journal,
            publication_year,
            language,
            authors_json,
            affiliations_json
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        rows,
    )

In [19]:
connection = connect_database()
create_tables(connection)

now = datetime.now(timezone.utc)

search_result = search_pubmed(PUBMED_QUERY, PUBMED_SAMPLE_SIZE)
esearch = search_result["esearchresult"]

run = {
    "run_id": now.strftime("%Y%m%dT%H%M%SZ"),
    "source": PUBMED_SOURCE,
    "query_text": PUBMED_QUERY,
    "query_variant": PUBMED_ACTIVE_VARIANT,
    "query_version": PUBMED_VARIANTS[PUBMED_ACTIVE_VARIANT]["version"],
    "retrieved_at": now.isoformat(),
    "total_results": int(esearch["count"]),
    "total_retrieved": len(esearch["idlist"]),
}

run

{'run_id': '20261016T122907Z',
 'source': 'pubmed',
 'query_text': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/01"[Date - Publication] : "2025/12/31"[Date - Publication] )',
 'query_variant': 'query_provisoria_atual',
 'query_version': '0.1',
 'retrieved_at': '2026-10-16T12:29:07.395786+00:00',
 'total_results': 100,
 'total_retrieved': 20}

In [20]:
pmids = esearch["idlist"]

search_path = RAW_PUBMED_DIR / f"{run['run_id']}_esearch.json"
search_path.write_text(
    json.dumps(search_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pubmed_xml = fetch_pubmed_citations(pmids)

fetch_path = RAW_PUBMED_DIR / f"{run['run_id']}_efetch.xml"
fetch_path.write_text(pubmed_xml, encoding="utf-8")

citations = parse_pubmed_xml(pubmed_xml)


search_snapshot = {
    "run_id": run["run_id"],
    "endpoint": "esearch",
    "payload_format": "json",
    "payload": search_result,
    "file_path": search_path,
}

fetch_snapshot = {
    "run_id": run["run_id"],
    "endpoint": "efetch",
    "payload_format": "xml",
    "payload": pubmed_xml,
    "file_path": fetch_path,
}

len(citations)

20

In [21]:
print("PMIDs retornados:", len(pmids))
print("Citações processadas:", len(citations))

PMIDs retornados: 20
Citações processadas: 20


In [22]:
insert_ingestion_run(connection, run)

insert_raw_snapshot(connection, search_snapshot)
insert_raw_snapshot(connection, fetch_snapshot)

insert_citations(connection, run["run_id"], citations)

connection.commit()